# Agent Evaluation Reporting

Load the saved evaluation outputs and display document-level, overall, and per-question tables.

In [8]:
import json
from collections.abc import Mapping
from pathlib import Path

import pandas as pd
from IPython.display import display

EVAL_ROOT = Path.cwd() / 'eval' if (Path.cwd() / 'eval').is_dir() else Path.cwd()
RESULTS_PATH = EVAL_ROOT / 'artifacts' / 'question_results.json'
SUMMARY_PATH = EVAL_ROOT / 'artifacts' / 'summary.json'

print(f'Results path: {RESULTS_PATH}')
print(f'Summary path: {SUMMARY_PATH}')

Results path: /Users/zach/Documents/GitHub/MMDocIR-indexer-agent-retrieval-project/eval/artifacts/question_results.json
Summary path: /Users/zach/Documents/GitHub/MMDocIR-indexer-agent-retrieval-project/eval/artifacts/summary.json


In [9]:
def _extract_feedback_map(
    answer_evaluation: Mapping[str, object] | None,
) -> dict[str, Mapping[str, object]]:
    if not isinstance(answer_evaluation, Mapping):
        return {}
    feedback_rows = answer_evaluation.get('feedback', [])
    if not isinstance(feedback_rows, list):
        return {}

    feedback_map: dict[str, Mapping[str, object]] = {}
    for feedback_row in feedback_rows:
        if not isinstance(feedback_row, Mapping):
            continue
        evaluator_key = feedback_row.get('evaluator_key')
        if isinstance(evaluator_key, str):
            feedback_map[evaluator_key] = feedback_row
    return feedback_map


ANSWER_EVALUATOR_KEYS = ('correctness', 'helpfulness')


def _feedback_value(
    feedback_map: Mapping[str, Mapping[str, object]],
    evaluator_key: str,
    field_name: str,
) -> object | None:
    feedback_row = feedback_map.get(evaluator_key, {})
    if not isinstance(feedback_row, Mapping):
        return None
    return feedback_row.get(field_name)


def _flatten_document_evaluator_summaries(
    document_metrics: list[dict[str, object]],
) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for document_metric in document_metrics:
        evaluator_summaries = document_metric.get('answer_evaluator_summaries', [])
        if not isinstance(evaluator_summaries, list):
            continue
        for evaluator_summary in evaluator_summaries:
            if not isinstance(evaluator_summary, Mapping):
                continue
            rows.append(
                {
                    'doc_name': document_metric.get('doc_name'),
                    'domain': document_metric.get('domain'),
                    'evaluator_key': evaluator_summary.get('evaluator_key'),
                    'question_count': evaluator_summary.get('question_count'),
                    'scored_question_count': evaluator_summary.get('scored_question_count'),
                    'true_rate': evaluator_summary.get('true_rate'),
                }
            )
    return pd.DataFrame(rows)


question_results = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))

document_metrics = summary['document_metrics']
overall_metrics = summary['overall_metrics']

document_evaluator_summary_df = _flatten_document_evaluator_summaries(document_metrics)
if not document_evaluator_summary_df.empty:
    document_evaluator_summary_df = document_evaluator_summary_df[
        document_evaluator_summary_df['evaluator_key'].isin(ANSWER_EVALUATOR_KEYS)
    ].reset_index(drop=True)

overall_evaluator_summary_df = pd.DataFrame(
    overall_metrics.get('answer_evaluator_summaries', [])
)
if not overall_evaluator_summary_df.empty and 'evaluator_key' in overall_evaluator_summary_df:
    overall_evaluator_summary_df = overall_evaluator_summary_df[
        overall_evaluator_summary_df['evaluator_key'].isin(ANSWER_EVALUATOR_KEYS)
    ].reset_index(drop=True)
question_results_df = pd.DataFrame(question_results)

answer_evaluation_series = question_results_df['answer_evaluation'].map(
    lambda value: value if isinstance(value, Mapping) else {}
)
feedback_map_series = answer_evaluation_series.map(_extract_feedback_map)
question_results_df['evaluator_error'] = answer_evaluation_series.map(
    lambda value: value.get('evaluator_error') if isinstance(value, Mapping) else None
)

for evaluator_key in ANSWER_EVALUATOR_KEYS:
    question_results_df[f'{evaluator_key}_score'] = feedback_map_series.map(
        lambda value, evaluator_key=evaluator_key: _feedback_value(
            value,
            evaluator_key,
            'score',
        )
    )
    question_results_df[f'{evaluator_key}_comment'] = feedback_map_series.map(
        lambda value, evaluator_key=evaluator_key: _feedback_value(
            value,
            evaluator_key,
            'comment',
        )
    )

## Question Level Analysis

In [10]:
drilldown_columns = [
    'doc_name',
    'question_index',
    'question',
    'expected_answer',
    'model_answer',
    'correctness_score',
    'helpfulness_score',
    'evaluator_error',
    'expected_pages',
    'retrieval_tool_call_count',
    'coarse_retrieved_pages',
    'reranked_pages',
    'initial_recall_at_k',
    'rerank_ndcg_at_k',
    'rerank_recall_at_k',
    'hit_rate_at_k',
]

question_results_df[drilldown_columns].sort_values(['doc_name', 'question_index'])

,doc_name,question_index,question,expected_answer,model_answer,correctness_score,helpfulness_score,evaluator_error,expected_pages,retrieval_tool_call_count,coarse_retrieved_pages,reranked_pages,initial_recall_at_k,rerank_ndcg_at_k,rerank_recall_at_k,hit_rate_at_k
6,0e94b4197b10096b1f4c699701570fbf.pdf,0,Where was Gestalt psychology concieved?,Berlin School of Experimental Psychology,Gestalt psychology was conceived at the **Berl...,True,True,NaN,[1],1,"[1, 0, 10, 7, 2, 6, 8, 9, 13, 5, 4, 11, 14, 12...","[1, 0, 10, 7, 2, 6, 8, 9, 13, 5]",1.0,1.0,1.0,1.0
7,0e94b4197b10096b1f4c699701570fbf.pdf,1,According to the chart on page 14 how much tim...,21%,Note that the chart appears on **page 13** of ...,True,True,NaN,[13],1,"[13, 11, 6, 8, 3, 5, 12, 0, 14, 4, 7, 9, 10, 2...","[13, 11, 6, 8, 3, 5, 12, 0, 14, 4]",1.0,1.0,1.0,1.0
8,0e94b4197b10096b1f4c699701570fbf.pdf,2,What range does red color represents in approx...,0-375 miles,Based on the retrieved chart from **`0e94b4197...,True,True,NaN,[9],1,"[9, 8, 3, 4, 5, 11, 13, 12, 14, 6, 0, 2, 1, 7,...","[9, 8, 3, 4, 5, 11, 13, 12, 14, 6]",1.0,1.0,1.0,1.0
9,0e94b4197b10096b1f4c699701570fbf.pdf,3,How does this document define the law of good ...,Elements of objects tend to be perceptually gr...,According to **page 10** of the document (*0e9...,True,True,NaN,[10],1,"[10, 0, 1, 7, 2, 8, 6, 14, 12, 3, 4, 13, 11, 9...","[10, 0, 1, 7, 2, 8, 6, 14, 12, 3]",1.0,1.0,1.0,1.0
10,0e94b4197b10096b1f4c699701570fbf.pdf,4,Which category has the most increase from 2005...,Eating out,Based on the infographic from **0e94b4197b1009...,True,True,NaN,[13],1,"[13, 11, 6, 12, 8, 14, 4, 9, 3, 10, 1, 5, 0, 7...","[13, 11, 6, 12, 8, 14, 4, 9, 3, 10]",1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38,reportq32015-151009093138-lva1-app6891_95.pdf,3,Which APP on APPOTA platform is top 10 Vietnam...,UC Browser Tiếng Việt,Based on the retrieved pages from the *Vietnam...,True,True,NaN,"[28, 29]",1,"[29, 28, 30, 31, 1, 32, 2, 26, 20, 3, 27, 23, ...","[29, 28, 30, 31, 1, 32, 2, 26, 20, 3]",1.0,1.0,1.0,1.0
39,reportq32015-151009093138-lva1-app6891_95.pdf,4,"In Q3 2015, what is the approximate range of c...","[1500, 8000]",,None,None,"Error code: 400 - {'type': 'error', 'error': {...","[25, 26]",0,[],[],0.0,0.0,0.0,0.0
40,reportq32015-151009093138-lva1-app6891_95.pdf,5,"As of Q3 2015, are there more active Instagram...",Appota,,None,None,"Error code: 400 - {'type': 'error', 'error': {...","[27, 32]",0,[],[],0.0,0.0,0.0,0.0
41,reportq32015-151009093138-lva1-app6891_95.pdf,6,"As of Q3 2015, is vietnam's adoption rate of i...","['lower', '38']",,None,None,"Error code: 400 - {'type': 'error', 'error': {...","[6, 14]",0,[],[],0.0,0.0,0.0,0.0


## Run Level Analysis

In [11]:
def _normalize_score_for_aggregation(score: object) -> float | None:
    if isinstance(score, bool):
        return float(score)
    if isinstance(score, int | float):
        return float(score)
    return None


def _question_type_label(question_type: object) -> str:
    if isinstance(question_type, list | tuple):
        labels = [str(item) for item in question_type if str(item)]
        return ' | '.join(labels) if labels else '(untyped)'
    if question_type is None:
        return '(untyped)'
    label = str(question_type)
    return label if label else '(untyped)'


def _build_run_level_analysis(
    question_df: pd.DataFrame,
    group_by: list[str],
) -> pd.DataFrame:
    valid_questions_df = question_df.loc[~question_df['has_evaluator_error']].copy()

    if group_by:
        all_questions_df = question_df.groupby(group_by, dropna=False).agg(
            question_count=('question', 'size'),
            error_excluded_question_count=('has_evaluator_error', 'sum'),
        )
        valid_summary_df = valid_questions_df.groupby(group_by, dropna=False).agg(
            evaluated_question_count=('question', 'size'),
            correctness_score_mean=('correctness_score_numeric', 'mean'),
            helpfulness_score_mean=('helpfulness_score_numeric', 'mean'),
            initial_recall_at_k_mean=('initial_recall_at_k', 'mean'),
            rerank_ndcg_at_k_mean=('rerank_ndcg_at_k', 'mean'),
            rerank_recall_at_k_mean=('rerank_recall_at_k', 'mean'),
            hit_rate_at_k_mean=('hit_rate_at_k', 'mean'),
            retrieval_tool_call_count_mean=('retrieval_tool_call_count', 'mean'),
        )
        run_level_analysis_df = all_questions_df.join(valid_summary_df, how='left').reset_index()
    else:
        run_level_analysis_df = pd.DataFrame(
            [
                {
                    'question_count': len(question_df),
                    'evaluated_question_count': len(valid_questions_df),
                    'error_excluded_question_count': int(question_df['has_evaluator_error'].sum()),
                    'correctness_score_mean': valid_questions_df['correctness_score_numeric'].mean(),
                    'helpfulness_score_mean': valid_questions_df['helpfulness_score_numeric'].mean(),
                    'initial_recall_at_k_mean': valid_questions_df['initial_recall_at_k'].mean(),
                    'rerank_ndcg_at_k_mean': valid_questions_df['rerank_ndcg_at_k'].mean(),
                    'rerank_recall_at_k_mean': valid_questions_df['rerank_recall_at_k'].mean(),
                    'hit_rate_at_k_mean': valid_questions_df['hit_rate_at_k'].mean(),
                    'retrieval_tool_call_count_mean': valid_questions_df['retrieval_tool_call_count'].mean(),
                }
            ]
        )

    run_level_analysis_df['evaluation_coverage'] = (
        run_level_analysis_df['evaluated_question_count']
        / run_level_analysis_df['question_count']
    )
    ordered_columns = [
        *group_by,
        'question_count',
        'evaluated_question_count',
        'error_excluded_question_count',
        'evaluation_coverage',
        'correctness_score_mean',
        'helpfulness_score_mean',
        'initial_recall_at_k_mean',
        'rerank_ndcg_at_k_mean',
        'rerank_recall_at_k_mean',
        'hit_rate_at_k_mean',
        'retrieval_tool_call_count_mean',
    ]
    return run_level_analysis_df[ordered_columns]


question_results_df['has_evaluator_error'] = question_results_df['evaluator_error'].fillna('').astype(str).str.len().gt(0)
question_results_df['question_type_label'] = question_results_df['question_type'].map(
    _question_type_label
)
for evaluator_key in ANSWER_EVALUATOR_KEYS:
    question_results_df[f'{evaluator_key}_score_numeric'] = question_results_df[
        f'{evaluator_key}_score'
    ].map(_normalize_score_for_aggregation)

full_run_analysis_df = _build_run_level_analysis(question_results_df, group_by=[])
domain_run_analysis_df = _build_run_level_analysis(question_results_df, group_by=['domain'])
document_run_analysis_df = _build_run_level_analysis(
    question_results_df,
    group_by=['domain', 'doc_name'],
)
question_type_run_analysis_df = _build_run_level_analysis(
    question_results_df,
    group_by=['question_type_label'],
)

display(full_run_analysis_df)

display(domain_run_analysis_df.sort_values(['domain'], kind='stable'))

display(document_run_analysis_df.sort_values(['domain', 'doc_name'], kind='stable'))

display(
    question_type_run_analysis_df.sort_values(
        ['question_type_label'],
        kind='stable',
    )
)

display(
    overall_evaluator_summary_df.sort_values(
        ['evaluator_key'],
        kind='stable',
    )
)

,question_count,evaluated_question_count,error_excluded_question_count,evaluation_coverage,correctness_score_mean,helpfulness_score_mean,initial_recall_at_k_mean,rerank_ndcg_at_k_mean,rerank_recall_at_k_mean,hit_rate_at_k_mean,retrieval_tool_call_count_mean
0,61,39,22,0.639344,0.897436,0.948718,0.940171,0.847075,0.940171,0.948718,1.102564


,domain,question_count,evaluated_question_count,error_excluded_question_count,evaluation_coverage,correctness_score_mean,helpfulness_score_mean,initial_recall_at_k_mean,rerank_ndcg_at_k_mean,rerank_recall_at_k_mean,hit_rate_at_k_mean,retrieval_tool_call_count_mean
0,Academic paper,18,NaN,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Research report / Introduction,24,20.0,4,0.833333,0.950000,0.950000,0.983333,0.888662,0.983333,1.000000,1.100000
2,Tutorial/Workshop,19,19.0,0,1.000000,0.842105,0.947368,0.894737,0.803299,0.894737,0.894737,1.105263


,domain,doc_name,question_count,evaluated_question_count,error_excluded_question_count,evaluation_coverage,correctness_score_mean,helpfulness_score_mean,initial_recall_at_k_mean,rerank_ndcg_at_k_mean,rerank_recall_at_k_mean,hit_rate_at_k_mean,retrieval_tool_call_count_mean
0,Academic paper,2310.05634v2.pdf,6,NaN,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Academic paper,2312.10997v5.pdf,6,NaN,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Academic paper,2401.18059v1.pdf,6,NaN,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Research report / Introduction,PH_2016.06.08_Economy-Final.pdf,6,6.0,0,1.0,1.000000,1.000000,0.944444,0.836964,0.944444,1.000000,1.000000
4,Research report / Introduction,earlybird-110722143746-phpapp02_95.pdf,5,5.0,0,1.0,0.800000,0.800000,1.000000,0.850293,1.000000,1.000000,1.200000
5,Research report / Introduction,fdac8d1e9ef56519371df7e6532df27d.pdf,5,5.0,0,1.0,1.000000,1.000000,1.000000,0.900000,1.000000,1.000000,1.200000
6,Research report / Introduction,reportq32015-151009093138-lva1-app6891_95.pdf,8,4.0,4,0.5,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
7,Tutorial/Workshop,0e94b4197b10096b1f4c699701570fbf.pdf,7,7.0,0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
8,Tutorial/Workshop,52b3137455e7ca4df65021a200aef724.pdf,6,6.0,0,1.0,0.666667,0.833333,0.666667,0.384475,0.666667,0.666667,1.166667
9,Tutorial/Workshop,ddoseattle-150627210357-lva1-app6891_95.pdf,6,6.0,0,1.0,0.833333,1.000000,1.000000,0.992638,1.000000,1.000000,1.166667


,question_type_label,question_count,evaluated_question_count,error_excluded_question_count,evaluation_coverage,correctness_score_mean,helpfulness_score_mean,initial_recall_at_k_mean,rerank_ndcg_at_k_mean,rerank_recall_at_k_mean,hit_rate_at_k_mean,retrieval_tool_call_count_mean
0,Chart,19,14.0,5,0.736842,1.000000,1.000000,1.000000,0.959334,1.000000,1.000000,1.000000
1,Chart | Generalized-text (Layout),1,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Chart | Table,1,1.0,0,1.000000,1.000000,1.000000,1.000000,0.630930,1.000000,1.000000,1.000000
3,Figure,11,9.0,2,0.818182,0.888889,1.000000,0.888889,0.774846,0.888889,0.888889,1.222222
4,Figure | Generalized-text (Layout),2,2.0,0,1.000000,0.500000,0.500000,1.000000,0.994803,1.000000,1.000000,0.500000
5,Generalized-text (Layout),3,3.0,0,1.000000,0.333333,0.666667,0.666667,0.263022,0.666667,0.666667,1.333333
6,Generalized-text (Layout) | Figure,5,2.0,3,0.400000,1.000000,1.000000,1.000000,0.815465,1.000000,1.000000,2.000000
7,Pure-text (Plain-text),11,6.0,5,0.545455,1.000000,1.000000,0.944444,0.945231,0.944444,1.000000,1.000000
8,Pure-text (Plain-text) | Table,2,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Table,5,2.0,3,0.400000,1.000000,1.000000,1.000000,0.959860,1.000000,1.000000,1.000000


,evaluator_key,source,question_count,scored_question_count,true_rate,numeric_score_mean,numeric_score_std
0,correctness,openevals,61,39,0.897436,None,None
1,helpfulness,openevals,61,39,0.948718,None,None


In [12]:
judge_comment_columns = [
    'doc_name',
    'question_index',
    'question',
    'correctness_comment',
    'helpfulness_comment',
    'evaluator_error',
]

question_results_df[judge_comment_columns].sort_values(['doc_name', 'question_index'])

,doc_name,question_index,question,correctness_comment,helpfulness_comment,evaluator_error
6,0e94b4197b10096b1f4c699701570fbf.pdf,0,Where was Gestalt psychology concieved?,The output correctly identifies the Berlin Sch...,The output directly addresses the core questio...,NaN
7,0e94b4197b10096b1f4c699701570fbf.pdf,1,According to the chart on page 14 how much tim...,The reference output indicates the correct ans...,The user is asking about a specific data point...,NaN
8,0e94b4197b10096b1f4c699701570fbf.pdf,2,What range does red color represents in approx...,The output states that the red color represent...,The user is asking about what range the red co...,NaN
9,0e94b4197b10096b1f4c699701570fbf.pdf,3,How does this document define the law of good ...,The response includes the core definition from...,Let me evaluate the output against the input q...,NaN
10,0e94b4197b10096b1f4c699701570fbf.pdf,4,Which category has the most increase from 2005...,"The output correctly identifies ""Eating out"" a...",The input asks which category has the most inc...,NaN
...,...,...,...,...,...,...
38,reportq32015-151009093138-lva1-app6891_95.pdf,3,Which APP on APPOTA platform is top 10 Vietnam...,The question asks which app on the APPOTA plat...,The input asks specifically which app(s) on th...,NaN
39,reportq32015-151009093138-lva1-app6891_95.pdf,4,"In Q3 2015, what is the approximate range of c...",NaN,NaN,"Error code: 400 - {'type': 'error', 'error': {..."
40,reportq32015-151009093138-lva1-app6891_95.pdf,5,"As of Q3 2015, are there more active Instagram...",NaN,NaN,"Error code: 400 - {'type': 'error', 'error': {..."
41,reportq32015-151009093138-lva1-app6891_95.pdf,6,"As of Q3 2015, is vietnam's adoption rate of i...",NaN,NaN,"Error code: 400 - {'type': 'error', 'error': {..."
